# L1: Automated Project: Planning, Estimation, and Allocation

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

## Initial Imports

In [5]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

import os
os.environ["LITELLM_ENDPOINT"] = "v1" # O forzar el comportamiento de la API estándar 
import yaml
from dotenv import load_dotenv
from crewai import Agent, Task, Crew

# Load environment variables
load_dotenv()

# Asegúrate de que la API key esté disponible para LiteLLM
# (LiteLLM busca automáticamente la variable GOOGLE_API_KEY en el entorno)
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# Configurar el LLM de Gemini usando el formato compatible con LiteLLM/CrewAI
# Reemplazamos gemini-1.5-flash por el identificador estándar actual (ej. gemini/gemini-2.5-flash)
# Configurar el LLM de Gemini con uno de los modelos disponibles en tu lista
llm_gemini = "gemini/gemini-3.5-flash"

<p style="background-color:#fff6ff; padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>. For more help, please see the <em>"Appendix - Tips and Help"</em> Lesson.</p>

## Set OpenAI Model

In [ ]:
#os.environ['OPENAI_MODEL_NAME'] = 'gemini/gemini-1.5-flash'

## Loading Tasks and Agents YAML files

In [6]:
# Define file paths for YAML configurations
files = {
    'agents': '../config/agents.yaml',
    'tasks': '../config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

## Create Pydantic Models for Structured Output

In [7]:
from typing import List
from pydantic import BaseModel, Field

class TaskEstimate(BaseModel):
    task_name: str = Field(..., description="Name of the task")
    estimated_time_hours: float = Field(..., description="Estimated time to complete the task in hours")
    required_resources: List[str] = Field(..., description="List of resources required to complete the task")

class Milestone(BaseModel):
    milestone_name: str = Field(..., description="Name of the milestone")
    tasks: List[str] = Field(..., description="List of task IDs associated with this milestone")

class ProjectPlan(BaseModel):
    tasks: List[TaskEstimate] = Field(..., description="List of tasks with their estimates")
    milestones: List[Milestone] = Field(..., description="List of project milestones")

## Create Crew, Agents and Tasks

In [8]:
# Configurar el LLM de Gemini correctamente para CrewAI
from crewai import Agent, Task, Crew
from langchain_google_genai import ChatGoogleGenerativeAI

# O bien usa directamente la nomenclatura compatible con LiteLLM que acepta CrewAI:
llm_gemini = "gemini/gemini-3.5-flash" # O la versión estable actual de tu entorno
# Creando los agentes usando el string del modelo compatible con LiteLLM
project_planning_agent = Agent(
    config=agents_config['project_planning_agent'],
    llm=llm_gemini,
    verbose=True
)

estimation_agent = Agent(
    config=agents_config['estimation_agent'],
    llm=llm_gemini,
    verbose=True
)

resource_allocation_agent = Agent(
    config=agents_config['resource_allocation_agent'],
    llm=llm_gemini,
    verbose=True
)

# Creating Tasks
task_breakdown = Task(
    config=tasks_config['task_breakdown'],
    agent=project_planning_agent
)

time_resource_estimation = Task(
    config=tasks_config['time_resource_estimation'],
    agent=estimation_agent
)

resource_allocation = Task(
    config=tasks_config['resource_allocation'],
    agent=resource_allocation_agent,
    output_json=ProjectPlan  # Aquí vinculamos el modelo Pydantic que creamos antes para estructurar la salida
)

# Creating Crew
crew = Crew(
  agents=[
    project_planning_agent,
    estimation_agent,
    resource_allocation_agent
  ],
  tasks=[
    task_breakdown,
    time_resource_estimation,
    resource_allocation
  ],
  verbose=True
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## Crew's Inputs

In [9]:
from IPython.display import display, Markdown

project = 'Website'
industry = 'Technology'
project_objectives = 'Create a website for a small business'
team_members = """
- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)
"""
project_requirements = """
- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust
"""

# Format the dictionary as Markdown for a better display in Jupyter Lab
formatted_output = f"""
**Project Type:** {project}

**Project Objectives:** {project_objectives}

**Industry:** {industry}

**Team Members:**
{team_members}
**Project Requirements:**
{project_requirements}
"""
# Display the formatted output as Markdown
display(Markdown(formatted_output))


**Project Type:** Website

**Project Objectives:** Create a website for a small business

**Industry:** Technology

**Team Members:**

- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)

**Project Requirements:**

- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust



## Kicking off the crew

In [10]:
# El diccionario con los datos de entrada
inputs = {
  'project_type': project,
  'project_objectives': project_objectives,
  'industry': industry,
  'team_members': team_members,
  'project_requirements': project_requirements
}

# Ejecutar la tripulación de forma asincrónica en el Notebook
result = await crew.kickoff_async(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e1e368da-c0bb-4808-830c-dba89ec239ec                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Carefully analyze the project_requirements for the Website project and break them down into individual   │
│  tasks. Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are      │
│  accounted for:                                                                                                 │
│                                                                                                                 │
│  - Create a responsive design that works well on desktop and mobile devices                                     │
│  - Implement a modern, visually appealing user interface with a clean look                                      │
│  - Develop a user-friendly navigation system with intuitive menu structure                                      │
│  - Include an "About Us" page highlighting the company's history and values                                     │
│  - Design a "Services" page showcasing the business's offerings with descriptions                               │
│  - Create a "Contact Us" page with a form and integrated map for communication                                  │
│  - Implement a blog section for sharing industry news and company updates                                       │
│  - Ensure fast loading times and optimize for search engines (SEO)                                              │
│  - Integrate social media links and sharing capabilities                                                        │
│  - Include a testimonials section to showcase customer feedback and build trust                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Team members:                                                                                                  │
│                                                                                                                 │
│  - John Doe (Project Manager)                                                                                   │
│  - Jane Doe (Software Engineer)                                                                                 │
│  - Bob Smith (Designer)                                                                                         │
│  - Alice Johnson (QA Engineer)                                                                                  │
│  - Tom Brown (QA Engineer)                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│  ID: 2bb29d41-f021-498b-bb38-fbaba775fbb5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Ultimate Project Planner                                                                            │
│                                                                                                                 │
│  Task: Carefully analyze the project_requirements for the Website project and break them down into individual   │
│  tasks. Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are      │
│  accounted for:                                                                                                 │
│                                                                                                                 │
│  - Create a responsive design that works well on desktop and mobile devices                                     │
│  - Implement a modern, visually appealing user interface with a clean look                                      │
│  - Develop a user-friendly navigation system with intuitive menu structure                                      │
│  - Include an "About Us" page highlighting the company's history and values                                     │
│  - Design a "Services" page showcasing the business's offerings with descriptions                               │
│  - Create a "Contact Us" page with a form and integrated map for communication                                  │
│  - Implement a blog section for sharing industry news and company updates                                       │
│  - Ensure fast loading times and optimize for search engines (SEO)                                              │
│  - Integrate social media links and sharing capabilities                                                        │
│  - Include a testimonials section to showcase customer feedback and build trust                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Team members:                                                                                                  │
│                                                                                                                 │
│  - John Doe (Project Manager)                                                                                   │
│  - Jane Doe (Software Engineer)                                                                                 │
│  - Bob Smith (Designer)                                                                                         │
│  - Alice Johnson (QA Engineer)                                                                                  │
│  - Tom Brown (QA Engineer)                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Using an approximate input token count for model 'gemini-3.5-flash': assuming one token per four characters because a native token count is unavailable.


[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Ultimate Project Planner                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Comprehensive Project Plan: Small Business Website Launch                                                    │
│                                                                                                                 │
│  **Prepared by:** John Doe, Lead Project Manager                                                                │
│  **Project:** Small Business Modern Website                                                                     │
│  **Project Duration:** 30 Business Days (6 Weeks)                                                               │
│  **Team Roster:**                                                                                               │
│  * **Project Manager:** John Doe (PM)                                                                           │
│  * **Software Engineer:** Jane Doe (SE)                                                                         │
│  * **Designer:** Bob Smith (DES)                                                                                │
│  * **QA Engineers:** Alice Johnson (QA1), Tom Brown (QA2)                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Executive Summary & Project Strategy                                                                        │
│                                                                                                                 │
│  As Project Manager, I have engineered this project plan to deliver a high-performing, modern, responsive       │
│  website tailored for small business growth. The plan addresses all 10 core business and technical              │
│  requirements, balancing visual sophistication with fast loading speeds, SEO best practices, seamless user      │
│  experience, and bulletproof functionality.                                                                     │
│                                                                                                                 │
│  Every task in this breakdown is mapped directly to team members, complete with clear scopes, timelines,        │
│  dependencies, and explicit deliverables to eliminate ambiguity and prevent scope creep.                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Task Breakdown Structure (WBS)                                                                              │
│                                                                                                                 │
│  ### Phase 1: Project Initiation, Architecture & Planning (Days 1 – 3)                                          │
│                                                                                                                 │
│  #### Task 1.1: Project Kickoff & Scope Baseline       

... (16 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resource Allocation Strategist                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  tasks=[TaskEstimate(task_name='Task 1.1: Project Kickoff & Scope Baseline', estimated_time_hours=16.67,        │
│  required_resources=['John Doe (PM)']), TaskEstimate(task_name='Task 1.2: Information Architecture & Sitemap    │
│  Specification', estimated_time_hours=24.67, required_resources=['John Doe (PM)', 'Bob Smith (DES)']),          │
│  TaskEstimate(task_name='Task 2.1: Brand Assets & UI Style Guide Development', estimated_time_hours=16.67,      │
│  required_resources=['Bob Smith (DES)']), TaskEstimate(task_name='Task 2.2: Wireframing Core Pages',            │
│  estimated_time_hours=18.67, required_resources=['Bob Smith (DES)']), TaskEstimate(task_name='Task 2.3:         │
│  High-Fidelity Responsive UI Design & Interactive Prototype', estimated_time_hours=27.0,                        │
│  required_resources=['Bob Smith (DES)']), TaskEstimate(task_name='Task 3.1: Environment Setup & Foundation      │
│  Architecture', estimated_time_hours=17.33, required_resources=['Jane Doe (SE)']),                              │
│  TaskEstimate(task_name='Task 3.2: Header, Navigation & Footer Components Implementation',                      │
│  estimated_time_hours=16.67, required_resources=['Jane Doe (SE)']), TaskEstimate(task_name='Task 3.3: Core      │
│  Static Pages Development ("About Us", "Services", "Testimonials")', estimated_time_hours=26.67,                │
│  required_resources=['Jane Doe (SE)']), TaskEstimate(task_name='Task 3.4: "Contact Us" Page & Interactive Map   │
│  Integration', estimated_time_hours=18.67, required_resources=['Jane Doe (SE)']), TaskEstimate(task_name='Task  │
│  3.5: Blog Engine & Article Layout Implementation', estimated_time_hours=18.67, required_resources=['Jane Doe   │
│  (SE)']), TaskEstimate(task_name='Task 3.6: Fast Loading & Technical SEO Optimization',                         │
│  estimated_time_hours=10.33, required_resources=['Jane Doe (SE)']), TaskEstimate(task_name='Task 4.1:           │
│  Copywriting & Media Content Entry', estimated_time_hours=20.33, required_resources=['John Doe (PM)', 'Jane     │
│  Doe (SE)']), TaskEstimate(task_name='Task 4.2: Social Media & Share Verification', estimated_time_hours=8.0,   │
│  required_resources=['Jane Doe (SE)']), TaskEstimate(task_name='Task 5.1: Responsive UI & Cross-Browser         │
│  Functional Testing', estimated_time_hours=24.67, required_resources=['Alice Johnson (QA1)']),                  │
│  TaskEstimate(task_name='Task 5.2: Performance, Accessibility & SEO Auditing', estimated_time_hours=8.67,       │
│  required_resources=['Alice Johnson (QA1)']), TaskEstimate(task_name='Task 5.3: Security, Form Validation &     │
│  Map Stress Testing', estimated_time_hours=16.67, required_resources=['Tom Brown (QA2)']),                      │
│  TaskEstimate(task_name='Task 5.4: Defect Remediation & Final Re-testing', estimated_time_hours=18.67,          │
│  required_resources=['Jane Doe (SE)', 'Bob Smith (DES)', 'Tom Brown (QA2)']), TaskEstimate(task_name='Task      │
│  6.1: Production Deployment & Domain/DNS Setup', estimated_time_hours=8.67, required_resources=['Jane Doe       │
│  (SE)', 'John Doe (PM)']), TaskEstimate(task_name='Task 6.2: Final Client Acceptance & Handover',               │
│  estimated_time_hours=8.67, required_resources=['John Doe (PM)'])] milestones=[Milestone(milestone_name='Phase  │
│  1: Project Initiation, Architecture & Planning', tasks

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Strategically allocate tasks for the Website project to team members based on their skills,              │
│  availability, and current workload. Ensure that each task is assigned to the most suitable team member and     │
│  that the workload is evenly distributed.                                                                       │
│                                                                                                                 │
│  Team members:                                                                                                  │
│                                                                                                                 │
│  - John Doe (Project Manager)                                                                                   │
│  - Jane Doe (Software Engineer)                                                                                 │
│  - Bob Smith (Designer)                                                                                         │
│  - Alice Johnson (QA Engineer)                                                                                  │
│  - Tom Brown (QA Engineer)                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│  Agent: Resource Allocation Strategist                                                                          │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e1e368da-c0bb-4808-830c-dba89ec239ec                                                                       │
│  Final Output: {"tasks":[{"task_name":"Task 1.1: Project Kickoff & Scope                                        │
│  Baseline","estimated_time_hours":16.67,"required_resources":["John Doe (PM)"]},{"task_name":"Task 1.2:         │
│  Information Architecture & Sitemap Specification","estimated_time_hours":24.67,"required_resources":["John     │
│  Doe (PM)","Bob Smith (DES)"]},{"task_name":"Task 2.1: Brand Assets & UI Style Guide                            │
│  Development","estimated_time_hours":16.67,"required_resources":["Bob Smith (DES)"]},{"task_name":"Task 2.2:    │
│  Wireframing Core Pages","estimated_time_hours":18.67,"required_resources":["Bob Smith                          │
│  (DES)"]},{"task_name":"Task 2.3: High-Fidelity Responsive UI Design & Interactive                              │
│  Prototype","estimated_time_hours":27.0,"required_resources":["Bob Smith (DES)"]},{"task_name":"Task 3.1:       │
│  Environment Setup & Foundation Architecture","estimated_time_hours":17.33,"required_resources":["Jane Doe      │
│  (SE)"]},{"task_name":"Task 3.2: Header, Navigation & Footer Components                                         │
│  Implementation","estimated_time_hours":16.67,"required_resources":["Jane Doe (SE)"]},{"task_name":"Task 3.3:   │
│  Core Static Pages Development (\"About Us\", \"Services\",                                                     │
│  \"Testimonials\")","estimated_time_hours":26.67,"required_resources":["Jane Doe (SE)"]},{"task_name":"Task     │
│  3.4: \"Contact Us\" Page & Interactive Map                                                                     │
│  Integration","estimated_time_hours":18.67,"required_resources":["Jane Doe (SE)"]},{"task_name":"Task 3.5:      │
│  Blog Engine & Article Layout Implementation","estimated_time_hours":18.67,"required_resources":["Jane Doe      │
│  (SE)"]},{"task_name":"Task 3.6: Fast Loading & Technical SEO                                                   │
│  Optimization","estimated_time_hours":10.33,"required_resources":["Jane Doe (SE)"]},{"task_name":"Task 4.1:     │
│  Copywriting & Media Content Entry","estimated_time_hours":20.33,"required_resources":["John Doe (PM)","Jane    │
│  Doe (SE)"]},{"task_name":"Task 4.2: Social Media & Share                                                       │
│  Verification","estimated_time_hours":8.0,"required_resources":["Jane Doe (SE)"]},{"task_name":"Task 5.1:       │
│  Responsive UI & Cross-Browser Functional Testing","estimated_time_hours":24.67,"required_resources":["Alice    │
│  Johnson (QA1)"]},{"task_name":"Task 5.2: Performance, Accessibility & SEO                                      │
│  Auditing","estimated_time_hours":8.67,"required_resources":["Alice Johnson (QA1)"]},{"task_name":"Task 5.3:    │
│  Security, Form Validation & Map Stress Testing","estimated_time_hours":16.67,"required_resources":["Tom Brown  │
│  (QA2)"]},{"task_name":"Task 5.4: Defect Remediation & Final                                                    │
│  Re-testing","estimated_time_hours":18.67,"required_resources":["Jane Doe (SE)","Bob Smith (DES)","Tom Brown    │
│  (QA2)"]},{"task_name":"Task 6.1: Production Deployment & Domain/DNS                                            │
│  Setup","estimated_time_hours":8.67,"required_resources":["Jane Doe (SE)","John Doe (PM)"]},{"task_name":"Task  │
│  6.2: Final Client Acceptance & Handover","estimated_t

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [2]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 10.5 MB/s  0:00:01m0:00:010:01


## Usage Metrics and Costs

Let’s see how much it would cost each time if this crew runs at scale.

In [11]:
import pandas as pd

costs = 0.150 * (crew.usage_metrics.prompt_tokens + crew.usage_metrics.completion_tokens) / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([crew.usage_metrics.dict()])
df_usage_metrics

Total costs: $0.0060


,total_tokens,prompt_tokens,cached_prompt_tokens,completion_tokens,reasoning_tokens,cache_creation_tokens,successful_requests
0,40025,18650,0,21375,6853,0,3


## Result

In [13]:
result.pydantic.model_dump()  # Pydantic v2

AttributeError: 'NoneType' object has no attribute 'model_dump'

## Inspect further

In [ ]:
tasks = result.pydantic.dict()['tasks']
df_tasks = pd.DataFrame(tasks)

# Display the DataFrame as an HTML table
df_tasks.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [('font-size', '120%')]}]
)

### Inspecting Milestones

In [ ]:
milestones = result.pydantic.dict()['milestones']
df_milestones = pd.DataFrame(milestones)

# Display the DataFrame as an HTML table
df_milestones.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [('font-size', '120%')]}]
)